# Notebook 5 — Walls on SAM-Refined Masks  (stage 5)

**Stage 5 — runs after Notebook 4.** This is Notebook 3's wall assignment re-applied to the
**SAM-refined** room masks instead of the watershed masks. It is **not** part of the default
local Run-All (`1 -> 2 -> 3`); run it intentionally once the Colab SAM stage (Notebook 4) has
produced `stage4_sam_refined/`.

> **Pipeline:** `1 occupancy -> 2 watershed -> 3 walls` (always, local) and, optionally,
> `4 SAM refinement (Colab) -> 5 walls on SAM masks`. Notebook 5 **fails loudly** if stage 4
> is missing — there is no auto-detect / silent fallback.

## Inputs
- `wallness.npy`, `transform.json` from `stage1_occupancy`.
- `room_labels_refined.npy`, `config.json` from `stage4_sam_refined` (Notebook 4 / Colab).
- the point cloud (reloaded with the same deterministic loader).

## Outputs  (`{out_root}/stage5_walls_sam_refined/`, zipped)
- per-room `room_XX_walls.ply`, `room_wall_masks.npz`, `room_labels.npy` (the refined labels
  used), `transform.json`, `config.json`.

### Setup
**Run-All ready.** Edit **`params.yaml`** (the only config surface), then run every cell top
to bottom — no cell edits, ever. `load_config()` reads it over the `Config` defaults.

> **Skipped `pip install -e .`?** Prepend this 2-line path-shim to the cell below so
> `import scan2bim` resolves from `notebooks/`:
> `import sys, os; sys.path.insert(0, os.path.abspath('..'))`

In [ ]:
# ====================== scan2bim setup (local) — Notebook 5 ==============================
# One loader replaces the old ~30-line bootstrap and the per-notebook config cell. With
# `pip install -e .`, `import scan2bim` works from anywhere; `load_config()` reads
# params.yaml (the ONLY file you edit) over the Config defaults and resolves paths.
import os
import numpy as np
import scan2bim
from scan2bim import artifacts as A, viz

CFG = scan2bim.load_config()        # params.yaml over Config defaults; file_path/out_root -> abs
SHOW_DEBUG = True                   # set False to skip the QA plots
print('scan2bim', scan2bim.__version__, 'loaded from', os.path.dirname(scan2bim.__file__))
print('input cloud :', CFG.file_path, '| exists:', os.path.isfile(CFG.file_path))
print('output root :', CFG.out_root)

### Step 1 — Load inputs (SAM-refined masks from stage 4)
Wallness + transform come from stage 1; the room masks come from the **SAM-refined** stage 4.
If stage 4 has not been produced, this fails with a clear "run Notebook 4 first" message.

In [ ]:
s1 = A.load_stage_dir(CFG.out_root, A.STAGE1)
wallness = A.load_npy(os.path.join(s1, A.WALLNESS_NPY)).astype(bool)
tf = A.load_transform(os.path.join(s1, A.TRANSFORM_JSON))

# Room masks come from the SAM-refined stage 4. Fail loudly if Notebook 4 hasn't run.
try:
    s4 = A.load_stage_dir(CFG.out_root, A.STAGE4)
except FileNotFoundError as e:
    raise RuntimeError(
        "Notebook 5 needs the SAM-refined masks from stage 4, which are missing. Run "
        "Notebook 4 (SAM refinement) in Colab first, then download stage4_sam_refined.zip "
        "into your local scan2bim_out/ and re-run this notebook.") from e

scan2bim.assert_upstream_config(CFG, A.load_stage_config(s1))   # same cloud + grid as stage 1
scan2bim.assert_upstream_config(CFG, A.load_stage_config(s4))   # ...and stage 4
labels = A.load_npy(os.path.join(s4, A.REFINED_LABELS_NPY)).astype('int32')
print('room masks from: SAM-refined (stage 4) | rooms:',
      len([r for r in np.unique(labels) if r >= 1]))

# Reload the cloud (deterministic -> aligns to tf) and assert it lands in the stage-1 grid.
pcd, pts = scan2bim.load_point_cloud(CFG)
scan2bim.assert_points_in_grid(pts, tf)

### Step 2 — Boundary-ring wall masks (identical to Notebook 3)

In [ ]:
wall_masks, dbg = scan2bim.room_wall_masks_boundary_ring(labels, wallness, CFG, return_debug=True)
print('rooms:', len(wall_masks), '| erode_px=%d  r_w(dilate_px)=%d' % (dbg['erode_px'], dbg['dilate_px']))

### Step 3 — Back-project the assigned wall pixels into 3-D

In [ ]:
band, floor_z, ceil_z = scan2bim.height_band_mask(pts, CFG, tf)
rooms3d = scan2bim.backproject_room_masks(pts, wall_masks, tf, keep_mask=band)
for e in rooms3d:
    print('room %02d: %7d wall points' % (e['room_id'], len(e['points'])))

### Optional — QA plot of the boundary-ring assignment

In [ ]:
if SHOW_DEBUG:
    viz.show_wall_assignment(labels, wall_masks, debug=dbg)

### Step 4 — Save per-room SAM-refined wall clouds + masks and package the ZIP

In [ ]:
import open3d as o3d
out_dir = A.ensure_dir(A.stage_dir(CFG.out_root, A.STAGE5))
A.save_room_wall_masks(os.path.join(out_dir, A.ROOM_WALL_MASKS_NPZ), wall_masks)

n_written = 0
for e in rooms3d:
    if len(e['points']) == 0:
        continue
    pc = o3d.geometry.PointCloud()
    pc.points = o3d.utility.Vector3dVector(e['points'])
    o3d.io.write_point_cloud(os.path.join(out_dir, 'room_%02d_walls.ply' % e['room_id']), pc)
    n_written += 1
print('wrote', n_written, 'SAM-refined room wall clouds')

A.save_npy(os.path.join(out_dir, A.ROOM_LABELS_NPY), labels)   # the refined labels we assigned on
A.save_transform(os.path.join(out_dir, A.TRANSFORM_JSON), tf,
                 extra=dict(floor_z=float(floor_z), ceil_z=float(ceil_z)))
A.save_config(os.path.join(out_dir, A.CONFIG_JSON), CFG)
zip_path = A.package_stage(CFG.out_root, A.STAGE5)
print('packaged ->', zip_path)